# Task 2: Hypothesis Testing

**Author:** Jozef Sládeček (xsladej00)

Goal: Test two hypotheses on traffic accident data in the Czech Republic.

In [13]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np

# Load data
accidents = pd.read_pickle('accidents.pkl.gz')
vehicles = pd.read_pickle('vehicles.pkl.gz')


---

## Hypothesis 1: Fatalities on 1st class vs. 3rd class roads

**Null hypothesis (H₀):** The probability of fatality in accidents is the same
on 1st class roads as on 3rd class roads.

**Method:** χ² test of independence (α = 0.05)

**Data:**
- `p36` = road type (1 = 1st class road, 3 = 3rd class road)
- `p9` = accident character (1 = with fatalities, 2 = property damage only)

In [14]:
roads = accidents[accidents['p36'].isin([1, 3])].copy()

print(f"Accidents on 1st and 3rd class roads: {len(roads)}")
print(f"\nDistribution by road type:")
print(roads['p36'].value_counts().sort_index())


Accidents on 1st and 3rd class roads: 66509

Distribution by road type:
p36
1    32123
3    34386
Name: count, dtype: int64


In [15]:
# Contingency table: p36 (road type) vs p9 (consequences)
contingency = pd.crosstab(roads['p36'], roads['p9'])

print("Contingency table:")
print(contingency)

Contingency table:
p9       1      2
p36              
1    10894  21229
3    10747  23639


In [16]:
# χ² test
chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f"  χ² test results:")
print(f"  χ² statistic: {chi2:.4f}")
print(f"  p-value: {p_value:.6f}")
print(f"  Degrees of freedom: {dof}")

  χ² test results:
  χ² statistic: 53.3884
  p-value: 0.000000
  Degrees of freedom: 1


In [17]:
# Compare observed and expected values
expected_df = pd.DataFrame(
    expected,
    index=['1st class', '3rd class'],
    columns=['p9=1 (death)', 'p9=2 (damage)']
)

print("Expected values (if H₀ holds):")
print(expected_df)

print("\nComparison:")
obs_1st_deaths = contingency.loc[1, 1]
exp_1st_deaths = expected[0, 0]

print(f"1ST CLASS - Deaths:")
print(f"  Observed: {obs_1st_deaths}")
print(f"  Expected: {exp_1st_deaths:.2f}")
print(f"  Difference: {obs_1st_deaths - exp_1st_deaths:+.2f}")

obs_3rd_deaths = contingency.loc[3, 1]
exp_3rd_deaths = expected[1, 0]

print(f"\n3RD CLASS - Deaths:")
print(f"  Observed: {obs_3rd_deaths}")
print(f"  Expected: {exp_3rd_deaths:.2f}")
print(f"  Difference: {obs_3rd_deaths - exp_3rd_deaths:+.2f}")

Expected values (if H₀ holds):
           p9=1 (death)  p9=2 (damage)
1st class  10452.327399   21670.672601
3rd class  11188.672601   23197.327399

Comparison:
1ST CLASS - Deaths:
  Observed: 10894
  Expected: 10452.33
  Difference: +441.67

3RD CLASS - Deaths:
  Observed: 10747
  Expected: 11188.67
  Difference: -441.67


### Conclusion of Hypothesis 1

**Decision (α = 0.05):**
- p-value = 0.000000 < 0.05
- **We reject H₀**

**Interpretation:**
- There is a **statistically significant higher** probability of fatality
  on 1st class roads compared to 3rd class roads.
- On 1st class roads, we observe **442 more deaths** than expected if road
  type had no effect.
- χ² = 53.39 indicates a large difference between observed and expected values.

---

### Part B: 1st class roads vs. highways

Using the `expected` output from the χ² test, we determine whether accidents
on 1st class roads lead more or less frequently to fatalities compared to highways.

**Comparison:**
- `p36 = 0`: highways (dálnice)
- `p36 = 1`: 1st class roads (silnice 1. třídy)
- `p9 = 1`: accident with fatalities
- `p9 = 2`: property damage only

In [18]:
roads_b = accidents[accidents['p36'].isin([0, 1])].copy()

print(f"Accidents on highways and 1st class roads: {len(roads_b)}")
print(f"\nDistribution by road type:")
print(roads_b['p36'].value_counts().sort_index())

Accidents on highways and 1st class roads: 43973

Distribution by road type:
p36
0    11850
1    32123
Name: count, dtype: int64


In [19]:
# Contingency table: p36 (road type) vs p9 (consequences)
contingency_b = pd.crosstab(roads_b['p36'], roads_b['p9'])

print("Contingency table (highways vs. 1st class):")
print(contingency_b)
print(f"\nTotals:")
print(f"  Highways: {contingency_b.loc[0].sum()}")
print(f"  1st class: {contingency_b.loc[1].sum()}")

Contingency table (highways vs. 1st class):
p9       1      2
p36              
0     1940   9910
1    10894  21229

Totals:
  Highways: 11850
  1st class: 32123


In [20]:
# χ² test for Part B
chi2_b, p_value_b, dof_b, expected_b = chi2_contingency(contingency_b)

print(f"📊 χ² test results (Part B):")
print(f"  χ² statistic: {chi2_b:.4f}")
print(f"  p-value: {p_value_b:.6f}")
print(f"  Degrees of freedom: {dof_b}")

📊 χ² test results (Part B):
  χ² statistic: 1288.0430
  p-value: 0.000000
  Degrees of freedom: 1


In [21]:
# Compare observed and expected values
expected_df_b = pd.DataFrame(
    expected_b,
    index=['Highways', '1st class'],
    columns=['p9=1 (death)', 'p9=2 (damage)']
)

print("Expected values (if H₀ holds):")
print(expected_df_b)

print("\nComparison - Deaths:")

# Highways
obs_highway_deaths = contingency_b.loc[0, 1]
exp_highway_deaths = expected_b[0, 0]

print(f"\nHIGHWAYS:")
print(f"  Observed: {obs_highway_deaths}")
print(f"  Expected: {exp_highway_deaths:.2f}")
print(f"  Difference: {obs_highway_deaths - exp_highway_deaths:+.2f}")

# 1st class roads
obs_1st_deaths = contingency_b.loc[1, 1]
exp_1st_deaths = expected_b[1, 0]

print(f"\n1ST CLASS ROADS:")
print(f"  Observed: {obs_1st_deaths}")
print(f"  Expected: {exp_1st_deaths:.2f}")
print(f"  Difference: {obs_1st_deaths - exp_1st_deaths:+.2f}")

# Interpretation
if obs_1st_deaths > exp_1st_deaths:
    print("\n✅ Accidents on 1st class roads lead MORE frequently to fatalities than on highways")
else:
    print("\n✅ Accidents on 1st class roads lead LESS frequently to fatalities than on highways")

Expected values (if H₀ holds):
           p9=1 (death)  p9=2 (damage)
Highways    3458.551839    8391.448161
1st class   9375.448161   22747.551839

Comparison - Deaths:

HIGHWAYS:
  Observed: 1940
  Expected: 3458.55
  Difference: -1518.55

1ST CLASS ROADS:
  Observed: 10894
  Expected: 9375.45
  Difference: +1518.55

✅ Accidents on 1st class roads lead MORE frequently to fatalities than on highways


### Conclusion of Part B

**Decision (α = 0.05):**
- p-value = 0.000000 < 0.05 (or >= 0.05)
- Reject H₀

**Interpretation:**
By comparing observed vs. expected values:
- On 1st class roads: 10894 deaths observed vs. 9375.45 expected
- On highways: 1940 deaths observed vs. 3458.55 expected

**Result:** Accidents on 1st class roads lead MORE frequently to
fatalities compared to highways

This analysis used the expected values from the χ² test to identify which
road type has higher fatality rates.

---

## Hypothesis 2: Damage by vehicle brand

**Null hypothesis (H₀):** The damage in accidents with brand X vehicles
is not significantly different from damage with brand Y vehicles.

**Method:** Mann-Whitney U test (α = 0.05)


**Why Mann-Whitney U test?**

- Damage distribution is highly skewed (skewness = 2.94)
- Mean is 2× higher than median → many outliers
- Mann-Whitney is non-parametric and robust for non-normal distributions

**Task:** Test two brand pairs:
- **Pair A:** Find brands where damage IS significantly different
- **Pair B:** Find brands where damage is NOT significantly different

**Data:**
- `p14*100` = damage amount (CZK) from accidents table
- `p45a` = vehicle brand code from vehicles table

In [22]:
brand_names = {
    26.0: 'MERCEDES',
    39.0: 'ŠKODA',
    47.0: 'VOLKSWAGEN'
}

### Pair A: ŠKODA vs. MERCEDES

**Expected outcome:** Damage IS significantly different

**Rationale:**
- MERCEDES is a premium luxury brand (significantly more expensive than ŠKODA)
- Higher repair costs for luxury vehicles
- Different accident patterns expected (highways vs. city)
- Sample sizes: MERCEDES (15,743 accidents), ŠKODA (92,151 accidents)

In [23]:
from scipy.stats import mannwhitneyu

# Merge and clean
merged = accidents[['p1', 'p14*100']].merge(vehicles[['p1', 'p45a']], on='p1', how='inner')
merged = merged.rename(columns={'p14*100': 'damage'})
merged_clean = merged[(merged['damage'] > 0) & (merged['damage'] < 10000000)].copy()

print(f"Records: {len(merged_clean):,}, Skewness: {merged_clean['damage'].skew():.2f}")

# Pair A: ŠKODA vs MERCEDES
print("\n" + "="*60)
print("PAIR A: ŠKODA vs. MERCEDES")
print("="*60)

skoda = merged_clean[merged_clean['p45a'] == 39.0]['damage']
mercedes = merged_clean[merged_clean['p45a'] == 26.0]['damage']

print(f"\nSample: ŠKODA {len(skoda):,}, MERCEDES {len(mercedes):,}")
print(f"ŠKODA - Mean: {skoda.mean():,.0f}, Median: {skoda.median():,.0f}")
print(f"MERCEDES - Mean: {mercedes.mean():,.0f}, Median: {mercedes.median():,.0f}")

stat, p = mannwhitneyu(skoda, mercedes, alternative='two-sided')
print(f"\nMann-Whitney: U={stat:,.0f}, p={p}")

if p < 0.05:
    diff = mercedes.median() - skoda.median()
    print(f" Reject H₀: MERCEDES higher (+{diff:,.0f} CZK)")
else:
    print(" Fail to reject H₀")

Records: 397,724, Skewness: 2.94

PAIR A: ŠKODA vs. MERCEDES

Sample: ŠKODA 92,151, MERCEDES 15,743
ŠKODA - Mean: 929,279, Median: 500,000
MERCEDES - Mean: 1,212,077, Median: 600,000

Mann-Whitney: U=658,760,722, p=4.759454411167622e-76
 Reject H₀: MERCEDES higher (+100,000 CZK)


---

### Pair B: ŠKODA vs. VOLKSWAGEN

**Expected outcome:** Damage is NOT significantly different

**Rationale:**
- Both brands belong to Volkswagen Group
- Share platforms and components (e.g., ŠKODA Octavia ≈ VW Golf platform)
- Similar price range and target market
- Comparable usage patterns (family cars, similar customer demographics)
- Expected similar repair costs and damage distributions
- Sample sizes: VOLKSWAGEN (36,509 accidents), ŠKODA (92,151 accidents)

In [24]:
print("\n" + "="*60)
print("PAIR B: ŠKODA vs. VOLKSWAGEN")
print("="*60)

skoda = merged_clean[merged_clean['p45a'] == 39.0]['damage']
vw = merged_clean[merged_clean['p45a'] == 47.0]['damage']

print(f"\nSample: ŠKODA {len(skoda):,}, VW {len(vw):,}")
print(f"ŠKODA - Mean: {skoda.mean():,.0f}, Median: {skoda.median():,.0f}")
print(f"VW - Mean: {vw.mean():,.0f}, Median: {vw.median():,.0f}")

stat, p = mannwhitneyu(skoda, vw, alternative='two-sided')
print(f"\nMann-Whitney: U={stat:,.0f}, p={p}")

if p < 0.05:
    diff = abs(vw.median() - skoda.median())
    higher = "VW" if vw.median() > skoda.median() else "ŠKODA"
    print(f"✅ Reject H₀: {higher} higher (+{diff:,.0f} CZK)")
else:
    print("❌ Fail to reject H₀: Similar distributions")


PAIR B: ŠKODA vs. VOLKSWAGEN

Sample: ŠKODA 92,151, VW 36,509
ŠKODA - Mean: 929,279, Median: 500,000
VW - Mean: 952,408, Median: 500,000

Mann-Whitney: U=1,674,850,786, p=0.22261215377356325
❌ Fail to reject H₀: Similar distributions
